# M02 — PromptTemplate 與結構化輸出

本 notebook 對應 `README.md`，逐格執行即可。

主軸：把 M01 手寫的訊息升級成「可重用的模板」，
再讓模型直接回傳乾淨的 Pydantic 物件（而不是一段要自己解析的文字）。

## 1. 環境準備

載入共用 helper，讓範例與供應商無關。
`get_model()` 會依 `.env` 的設定回傳一個 chat model（OpenAI / Anthropic / Ollama 皆可）。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 回顧 M01：手寫訊息

先回顧上一個模組的做法：手動組一串訊息再 invoke。
這能跑，但 system 指令寫死、使用者輸入要用 f-string 拼，重用性差。

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

question = "誰是秦始皇？"
messages = [
    SystemMessage("你是一位歷史老師，回答要精簡。"),
    HumanMessage(question),
]
response = model.invoke(messages)
print(response.content)
# Expected output: 一段精簡的、關於秦始皇的中文說明。

## 3. 第一個模板：system + human

把上面那串訊息升級成有 `{變數}` 洞的模板。
`("system", "...")` 對應 `SystemMessage`，`("human", "...")` 對應 `HumanMessage`，
差別是現在可以填變數、可以重用。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位{role}，回答要精簡。"),
        ("human", "{question}"),
    ]
)

# invoke 模板會「填洞」並產出一串訊息（就是 M01 那種訊息 list）。
filled = prompt.invoke({"role": "歷史老師", "question": "誰是秦始皇？"})
print(filled.to_messages())
# Expected output: [SystemMessage('你是一位歷史老師，回答要精簡。'), HumanMessage('誰是秦始皇？')]

# 把填好的訊息送進模型。
response = model.invoke(filled)
print(response.content)
# Expected output: 一段精簡的、關於秦始皇的中文說明。

## 4. partial：先固定一部分變數

有些變數啟動時就確定了（例如老師的角色），不必每次都傳。
`partial` 先填好 role，之後只要傳 question。

In [ ]:
history_teacher_prompt = prompt.partial(role="歷史老師")

# 現在只需要傳 question，role 已固定。
filled = history_teacher_prompt.invoke({"question": "什麼是焚書坑儒？"})
print(filled.to_messages())
# Expected output: [SystemMessage('你是一位歷史老師，回答要精簡。'), HumanMessage('什麼是焚書坑儒？')]

## 5. MessagesPlaceholder：把對話歷史注入模板

對話歷史是「一串訊息」，不是字串。
`MessagesPlaceholder("history")` 在模板裡預留一個位置，invoke 時塞進一個訊息 list。
這就是 M01 訊息列表的延伸：固定指令與動態歷史分開管理。

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import AIMessage

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是親切的客服助理。"),
        MessagesPlaceholder("history"),   # a list of past messages goes here
        ("human", "{question}"),
    ]
)

# Past turns: reuse M01-style HumanMessage / AIMessage objects.
history = [
    HumanMessage("我上週買的耳機收不到包裹。"),
    AIMessage("方便提供您的訂單編號嗎？我幫您查詢。"),
]

filled = chat_prompt.invoke(
    {"history": history, "question": "訂單編號是 A123。"}
)
for m in filled.to_messages():
    print(type(m).__name__, ":", m.content)
# Expected output (4 行):
# SystemMessage : 你是親切的客服助理。
# HumanMessage : 我上週買的耳機收不到包裹。
# AIMessage : 方便提供您的訂單編號嗎？我幫您查詢。
# HumanMessage : 訂單編號是 A123。

response = model.invoke(filled)
print(response.content)
# Expected output: 模型在「記得前文」的情況下，針對訂單 A123 的回覆。

## 6. few-shot：把範例寫進模板

要模型照特定風格回答，最有效的方式是給它幾個範例。
最直接的做法：把「範例問 / 範例答」當成額外的 human / ai 訊息放進模板，
模型看到示範就會模仿。

In [ ]:
fewshot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "把使用者輸入的句子改寫成正式書面語。"),
        ("human", "這東西超讚的啦"),
        ("ai", "此產品表現優異。"),          # demonstration 1
        ("human", "他超會講"),
        ("ai", "他的口才十分出色。"),          # demonstration 2
        ("human", "{sentence}"),
    ]
)

filled = fewshot_prompt.invoke({"sentence": "這家店的服務爛透了"})
response = model.invoke(filled)
print(response.content)
# Expected output: 一句正式書面語，例如「這家店的服務品質有待加強。」

## 7. 結構化輸出（一）：資訊抽取

到目前為止輸出都是文字，得自己解析。
用 Pydantic `BaseModel` 定義想要的形狀，再用 `with_structured_output` 包住模型，
invoke 就會直接回傳那個物件實例。
`Field(description=...)` 會變成給模型的提示，幫它抓對欄位。

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """A person mentioned in the sentence."""
    name: str = Field(description="人物的名字")
    age: int = Field(description="人物的年齡（歲）")

structured_model = model.with_structured_output(Person)
person = structured_model.invoke("小明今年 10 歲，住在台北。")
print(person)
print("name =", person.name, "| age =", person.age)
# Expected output:
# name='小明' age=10
# name = 小明 | age = 10

## 8. 結構化輸出（二）：分類任務

分類時用 `Literal` 把答案限制在固定選項裡，模型就不會自由發揮回一個沒預期的類別。
這裡把客服訊息分類，並回傳一個結構化結果（含類別 + 是否緊急 + 一句理由）。

In [ ]:
from typing import Literal

class TicketTriage(BaseModel):
    """Classification result for a customer support message."""
    category: Literal["退貨", "技術問題", "帳務", "一般詢問"] = Field(
        description="這則訊息屬於哪一類"
    )
    urgent: bool = Field(description="是否需要優先處理")
    reason: str = Field(description="做出此分類的一句話理由")

classifier = model.with_structured_output(TicketTriage)
result = classifier.invoke("我的帳戶被重複扣款兩次，請馬上處理！")
print(result)
# Expected output (示意):
# category='帳務' urgent=True reason='訊息提到重複扣款並要求立即處理。'

## 9. 組合：模板 + 結構化輸出

真實用法常常是「模板組輸入」加「結構化拿輸出」。
先用模板把任務說清楚（含變數），再把它接到結構化模型上。
（M03 會教用 `|` 把這兩步串成一條正式管線，這裡先手動分兩步示範概念。）

In [ ]:
extract_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "從句子中抽取人物資訊。"),
        ("human", "{sentence}"),
    ]
)

filled = extract_prompt.invoke({"sentence": "老王是一位 55 歲的工程師。"})
person = structured_model.invoke(filled)   # reuse the Person-structured model
print(person)
# Expected output: name='老王' age=55

## 🧪 練習 1：擴充抽取的欄位

把 `Person` 多加一個欄位 `city: str`（用 `Field(description=...)` 寫清楚說明），
然後用句子「小華今年 22 歲，在高雄念書。」測試。
觀察模型是否同時抓到 name / age / city。

提示：改完 `Person` 後要重新 `model.with_structured_output(Person)` 拿新的結構化模型。

In [ ]:
# TODO: 在這裡寫你的答案
# class Person(BaseModel):
#     name: str = Field(description="人物的名字")
#     age: int = Field(description="人物的年齡（歲）")
#     city: str = Field(description="人物所在的城市")
#
# structured_model = model.with_structured_output(Person)
# print(structured_model.invoke("小華今年 22 歲，在高雄念書。"))

## 🧪 練習 2：給分類器加 few-shot 引導

在 `extract_prompt` 或一個新的分類模板裡，加入 1～2 組 few-shot 範例訊息
#（`("human", "範例輸入")` 後接 `("ai", "範例輸出")`），
觀察分類結果是否更穩定、更符合你想要的判準。

想一想：few-shot 範例放在 system 之後、真正的 `{變數}` 之前，為什麼是合理的位置？

In [ ]:
# TODO: 在這裡寫你的答案

## 小結 & 下一步

這個模組你做到了：

- 用 `ChatPromptTemplate.from_messages` 把手寫訊息升級成可重用模板，並用變數填充。
- 用 `partial` 預填固定變數、用 `MessagesPlaceholder` 注入對話歷史。
- 用 few-shot 範例訊息引導輸出風格。
- 用 Pydantic + `with_structured_output` 做資訊抽取與分類，直接拿到物件而非文字。

下一個模組 **M03 — LCEL 與 Runnable 管線** 會把 `prompt`、`model`、輸出解析
用 `|` 串成一條管線（`prompt | model | parser`），讓整個流程變成一個
可組合、可並行的物件，不必再像第 9 格那樣手動分兩步呼叫。